In [ ]:
#| default_exp convert

## Converting Python into notebooks

Move Python modules into nbdev notebooks while preserving an exportable story.

Conversion is a bridge for projects that began as `.py` files but want nbdev notebooks as the source of truth. The converter keeps the first pass mechanical: preserve source order, create explicit `#| default_exp` and `#| export` cells, and split large classes only when that makes future notebook edits easier.

```python
py2nb("src/tool.py", nbs_path="nbs")
py2nbs("src", nbs_path="nbs", preserve_tree=True)
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from fastcore.nbio import read_nb as _read_nb
from nbskill.convert import py2nb as _example_py2nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
root = demo_path("05_convert_example")
try:
    root.mkdir()
    src = root / "calc.py"
    dest = root / "calc.ipynb"
    src.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
    _example_py2nb(str(src), dest=str(dest))
    _preview = [cell.source.splitlines()[0] for cell in _read_nb(dest).cells]
finally:
    remove_demo_path(root)
_preview

Wrote 2 cells to nbs/data/05_convert_example/calc.ipynb


['#| default_exp calc', '#| export']

In [ ]:
#| export
import ast,copy
import fnmatch
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import call_parse

from nbskill.foundation import cli_return, is_definition_node, node_start_line, tracked_call

### Slicing Python source with `ast`

Conversion starts by finding exact source ranges for imports, functions, classes, and methods. Using `ast` keeps the conversion structured instead of relying on fragile string searches.

In [ ]:
#| export
def _node_source(lines, node):
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1
    return "\n".join(lines[start:node.end_lineno]).strip("\n")

In [ ]:
#| export
def _node_line_count(node):
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]])
    return node.end_lineno - start + 1

In [ ]:
#| export
def _patchable_method(node):
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return False
    if node.decorator_list: return False
    return bool(node.args.posonlyargs or node.args.args)

In [ ]:
#| export
def _annotate_first_arg(node, class_name):
    args = node.args.posonlyargs or node.args.args
    if not args: return
    args[0].annotation = ast.Name(id=class_name, ctx=ast.Load())

In [ ]:
#| export
def _patch_method_source(method, class_name):
    node = copy.deepcopy(method)
    node.decorator_list = []
    _annotate_first_arg(node, class_name)
    ast.fix_missing_locations(node)
    return f"@patch\n{ast.unparse(node)}"

In [ ]:
#| export
def _class_without_methods(lines, cls, methods):
    class_start = min([cls.lineno, *[d.lineno for d in cls.decorator_list]])
    class_line = cls.lineno - class_start
    src_lines = lines[class_start - 1:cls.end_lineno]
    remove_ranges = []
    for method in methods:
        start = min([method.lineno, *[d.lineno for d in method.decorator_list]]) - class_start
        stop = method.end_lineno - class_start + 1
        remove_ranges.append((start, stop))
    for start, stop in sorted(remove_ranges, reverse=True): del src_lines[start:stop]
    body = src_lines[class_line + 1:]
    if not any(line.strip() for line in body): src_lines.append("    pass")
    return "\n".join(src_lines).strip("\n")

In [ ]:
#| export
def _export_cell(source):
    return mk_cell(f"#| export\n{source.strip()}")

### Assembling nbdev cells

The converter writes `#| default_exp` first, collects imports together, skips existing `__all__` declarations, and emits exported cells for the definitions that should become the notebook source.

In [ ]:
#| export
def _stable_cell_id(default_exp, idx, source):
    raw = f"{default_exp}:{idx}:{source}".encode("utf-8")
    return f"py2nb-{hashlib.sha1(raw).hexdigest()[:8]}"


def _stabilize_cells(cells, default_exp):
    for idx, cell in enumerate(cells):
        cell.id = _stable_cell_id(default_exp, idx, getattr(cell, "source", ""))
    return cells


def _is_module_docstring(node):
    return isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str)


def _py2nb_cells(source, default_exp, class_lines=100, method_lines=10):
    tree = ast.parse(source)
    lines = source.splitlines()
    cells = [mk_cell(f"#| default_exp {default_exp}")]
    pending_imports = []
    needs_patch = False

    if tree.body and _is_module_docstring(tree.body[0]):
        cells.append(mk_cell(tree.body[0].value.value.strip(), cell_type="markdown"))

    def flush_imports():
        if pending_imports:
            cells.append(_export_cell("\n".join(pending_imports)))
            pending_imports.clear()

    for idx, node in enumerate(tree.body):
        if idx == 0 and _is_module_docstring(node): continue
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            pending_imports.append(_node_source(lines, node))
            continue
        if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == "__all__" for target in node.targets):
            continue
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == "__all__":
            continue
        if isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name) and node.target.id == "__all__":
            continue
        flush_imports()
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): cells.append(_export_cell(_node_source(lines, node)))
        elif isinstance(node, ast.ClassDef):
            methods = [child for child in node.body if _patchable_method(child) and _node_line_count(child) > method_lines]
            if _node_line_count(node) > class_lines and methods:
                needs_patch = True
                cells.append(_export_cell(_class_without_methods(lines, node, methods)))
                for method in methods: cells.append(_export_cell(_patch_method_source(method, node.name)))
            else: cells.append(_export_cell(_node_source(lines, node)))
        else:
            cells.append(_export_cell(_node_source(lines, node)))
    flush_imports()
    if needs_patch: cells.insert(1, _export_cell("from fastcore.basics import patch"))
    return _stabilize_cells(cells, default_exp)

In [ ]:
#| export
def _module_name_for_path(path, module_base=None):
    pth = Path(path)
    if module_base is None: return pth.stem
    rel = pth.relative_to(module_base).with_suffix("")
    return ".".join(rel.parts)


def _py2nb_file(path, nbs_path="nbs", dest=None, class_lines=100, method_lines=10, default_exp=None, dry_run=False, force=True):
    pth = Path(path)
    source = pth.read_text(encoding="utf-8")
    default_exp = default_exp or pth.stem
    out_path = Path(dest) if dest else Path(nbs_path) / f"{default_exp.replace('.', '/')}.ipynb"
    if out_path.exists() and not force:
        raise FileExistsError(f"Refusing to overwrite existing notebook: {out_path}")
    nb = new_nb(_py2nb_cells(source, default_exp, class_lines=class_lines, method_lines=method_lines))
    if not dry_run:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        _write_nb(nb, out_path)
    return {
        "source": str(pth),
        "notebook": str(out_path),
        "default_exp": default_exp,
        "cell_count": len(nb.cells),
        "dry_run": dry_run,
    }

### Public conversion commands

`py2nb` handles either one Python file or a folder of Python files. `py2nbs` remains as the explicit folder-oriented wrapper for callers that already use that name.

In [ ]:
#| export
_DEFAULT_IGNORE_DIRS = {
    "__pycache__", ".git", ".hg", ".mypy_cache", ".pytest_cache", ".ruff_cache",
    ".tox", ".venv", "venv", "env", "build", "dist", ".ipynb_checkpoints",
    "node_modules",
}
_DEFAULT_IGNORE_FILES = {"__init__.py"}
_TEST_DIRS = {"test", "tests", "testing"}


def _patterns(value):
    if not value: return []
    if isinstance(value, (list, tuple, set)): return [str(item) for item in value if str(item)]
    return [item.strip() for item in str(value).split(",") if item.strip()]


def _matches_any(path, patterns):
    text = path.as_posix()
    return any(fnmatch.fnmatch(text, pat) or fnmatch.fnmatch(path.name, pat) for pat in patterns)


def _package_dirs(root):
    return sorted(
        path for path in root.iterdir()
        if path.is_dir() and (path / "__init__.py").exists() and path.name not in _DEFAULT_IGNORE_DIRS
    )


def _resolve_source_tree(path, package=None):
    root = Path(path)
    if root.is_file(): return root, root.parent, root.parent
    if package:
        for candidate in (root / package, root / "src" / package, root):
            if candidate.exists() and candidate.name == package:
                return candidate, candidate.parent, root
    if root.name == "src" and _package_dirs(root):
        pkgs = _package_dirs(root)
        return (pkgs[0], root, root) if len(pkgs) == 1 else (root, root, root)
    if (root / "src").is_dir() and _package_dirs(root / "src"):
        pkgs = _package_dirs(root / "src")
        return (pkgs[0], root / "src", root) if len(pkgs) == 1 else (root / "src", root / "src", root)
    if (root / "__init__.py").exists(): return root, root.parent, root
    pkgs = _package_dirs(root)
    if len(pkgs) == 1: return pkgs[0], root, root
    return root, root, root


def _is_ignored_py(path, rel, include=None, exclude=None, skip_init=True, include_tests=False):
    if any(part in _DEFAULT_IGNORE_DIRS or part.startswith(".") for part in rel.parts[:-1]): return True
    if skip_init and path.name in _DEFAULT_IGNORE_FILES: return True
    if not include_tests and (set(rel.parts[:-1]) & _TEST_DIRS or path.name.startswith("test_") or path.name.endswith("_test.py")): return True
    if include and not _matches_any(rel, include): return True
    if exclude and _matches_any(rel, exclude): return True
    return False


def _python_files(scan_root, recursive=True, maxdepth=None):
    files = scan_root.rglob("*.py") if recursive else scan_root.glob("*.py")
    for path in sorted(files):
        rel = path.relative_to(scan_root)
        if maxdepth is not None and len(rel.parts) - 1 > maxdepth: continue
        yield path


def _dest_for_module(nbs_path, default_exp, preserve_tree):
    name = f"{default_exp.replace('.', '/')}.ipynb" if preserve_tree else f"{default_exp.rsplit('.', 1)[-1]}.ipynb"
    return Path(nbs_path) / name


def _py2nbs_folder(
    path, nbs_path="nbs", recursive=True, maxdepth=None, preserve_tree=True,
    class_lines=100, method_lines=10, package=None, include=None, exclude=None,
    skip_init=True, include_tests=False, dry_run=False, force=True,
):
    scan_root, module_base, project_root = _resolve_source_tree(path, package=package)
    include, exclude = _patterns(include), _patterns(exclude)
    report = {
        "source_root": str(project_root),
        "scan_root": str(scan_root),
        "module_base": str(module_base),
        "nbs_path": str(nbs_path),
        "converted": [],
        "skipped": [],
        "blockers": [],
        "dry_run": dry_run,
    }
    seen_dests = {}
    for pth in _python_files(scan_root, recursive=recursive, maxdepth=maxdepth):
        rel = pth.relative_to(scan_root)
        if _is_ignored_py(pth, rel, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests):
            report["skipped"].append(str(pth))
            continue
        default_exp = _module_name_for_path(pth, module_base=module_base)
        dest = _dest_for_module(nbs_path, default_exp, preserve_tree=preserve_tree)
        if str(dest) in seen_dests:
            report["blockers"].append(f"Notebook collision: {pth} and {seen_dests[str(dest)]} -> {dest}")
            continue
        seen_dests[str(dest)] = str(pth)
        item = _py2nb_file(
            str(pth), nbs_path=nbs_path, dest=str(dest), class_lines=class_lines,
            method_lines=method_lines, default_exp=default_exp, dry_run=dry_run, force=force,
        )
        report["converted"].append(item)
        print(f"{'Would write' if dry_run else 'Wrote'} {item['cell_count']} cells to {item['notebook']}")
    print(f"{'Would convert' if dry_run else 'Converted'} {len(report['converted'])} Python files to {nbs_path}")
    if report["skipped"]: print(f"Skipped {len(report['skipped'])} ignored Python files")
    if report["blockers"]: print(f"Blocked {len(report['blockers'])} conversion item(s)")
    return report


@call_parse
@tracked_call
def py2nb(
    path: str,  # Python file or folder to convert
    nbs_path: str = "nbs",  # Folder for generated notebooks
    dest: str | None = None,  # Explicit notebook path for one file, or output folder for a directory
    recursive: bool = True,  # Search subfolders when path is a folder
    maxdepth: int | None = None,  # Maximum folder depth to search
    preserve_tree: bool = True,  # Preserve folder structure below nbs_path for folder conversion
    class_lines: int = 100,  # Split methods out of classes longer than this
    method_lines: int = 10,  # Split methods longer than this out of large classes
    package: str | None = None,  # Package name to select inside a project root
    include: str | None = None,  # Comma-separated include glob(s)
    exclude: str | None = None,  # Comma-separated exclude glob(s)
    skip_init: bool = True,  # Skip __init__.py by default
    include_tests: bool = False,  # Convert test files too
    dry_run: bool = False,  # Plan conversion without writing notebooks
    force: bool = True,  # Overwrite existing generated notebooks
):
    "Convert a Python file or folder into nbdev-style notebooks using AST parsing."
    pth = Path(path)
    if pth.is_dir():
        out_root = dest or nbs_path
        report = _py2nbs_folder(
            pth, nbs_path=out_root, recursive=recursive, maxdepth=maxdepth,
            preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
            package=package, include=include, exclude=exclude, skip_init=skip_init,
            include_tests=include_tests, dry_run=dry_run, force=force,
        )
        return cli_return(report)

    default_exp = _module_name_for_path(pth)
    report = _py2nb_file(
        path, nbs_path=nbs_path, dest=dest, class_lines=class_lines,
        method_lines=method_lines, default_exp=default_exp, dry_run=dry_run, force=force,
    )
    print(f"{'Would write' if dry_run else 'Wrote'} {report['cell_count']} cells to {report['notebook']}")
    return cli_return(report)

In [ ]:
root = demo_path("05_convert_tests")
try:
    root.mkdir()
    src = root / "demo.py"
    nbs_path = root / "nbs"
    src.write_text("def add(a, b):\n    return a + b\n", encoding="utf-8")
    py2nb(str(src), nbs_path=str(nbs_path))
    nb = _read_nb(nbs_path / "demo.ipynb")
    assert any("#| default_exp demo" in cell.source for cell in nb.cells)
    assert any("def add" in cell.source for cell in nb.cells)

    pkg = root / "pkg"
    (pkg / "sub").mkdir(parents=True)
    (pkg / "one.py").write_text("def one():\n    return 1\n", encoding="utf-8")
    (pkg / "sub" / "two.py").write_text("def two():\n    return 2\n", encoding="utf-8")
    all_nbs = root / "all_nbs"
    py2nb(str(pkg), nbs_path=str(all_nbs))
    assert (all_nbs / "one.ipynb").exists()
    assert (all_nbs / "sub" / "two.ipynb").exists()
finally:
    remove_demo_path(root)

In [ ]:
#| export
@call_parse
@tracked_call
def py2nbs(
    path: str,  # Folder containing Python files
    nbs_path: str = "nbs",  # Folder for generated notebooks
    recursive: bool = True,  # Search subfolders
    maxdepth: int | None = None,  # Maximum folder depth to search
    preserve_tree: bool = True,  # Preserve folder structure below nbs_path
    class_lines: int = 100,  # Split methods out of classes longer than this
    method_lines: int = 10,  # Split methods longer than this out of large classes
    package: str | None = None,  # Package name to select inside a project root
    include: str | None = None,  # Comma-separated include glob(s)
    exclude: str | None = None,  # Comma-separated exclude glob(s)
    skip_init: bool = True,  # Skip __init__.py by default
    include_tests: bool = False,  # Convert test files too
    dry_run: bool = False,  # Plan conversion without writing notebooks
    force: bool = True,  # Overwrite existing generated notebooks
):
    "Convert all Python files in a folder into nbdev-style notebooks."
    report = _py2nbs_folder(
        path, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
        preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
        package=package, include=include, exclude=exclude, skip_init=skip_init,
        include_tests=include_tests, dry_run=dry_run, force=force,
    )
    return cli_return(report)

In [0]:
#| export
def _primary_package_name(source, package=None):
    scan_root, module_base, _ = _resolve_source_tree(source, package=package)
    if package: return package
    if (scan_root / "__init__.py").exists(): return scan_root.name
    pkgs = _package_dirs(scan_root)
    if len(pkgs) == 1: return pkgs[0].name
    return Path(source).resolve().name.replace("-", "_")

In [0]:
#| export
def _write_if_allowed(path, text, dry_run=True, force=False):
    pth = Path(path)
    if dry_run: return {"path": str(pth), "action": "would-write"}
    if pth.exists() and not force: return {"path": str(pth), "action": "exists"}
    pth.parent.mkdir(parents=True, exist_ok=True)
    pth.write_text(text, encoding="utf-8")
    return {"path": str(pth), "action": "wrote"}

In [0]:
#| export
def _nbdev_pyproject_text(name, package, nbs_path):
    return f"""[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "{name}"
version = "0.0.1"
description = "Converted nbdev project"
requires-python = ">=3.10"
dependencies = ["nbdev>=3.0.15"]

[tool.setuptools.packages.find]
include = ["{package}*"]

[tool.nbdev]
nbs_path = "{nbs_path}"
lib_path = "{package}"
doc_path = "_docs"
"""

In [ ]:
#| export
def _validate_nbdev_project(dest, package, run_validation=True, dry_run=True):
    result = {"requested": run_validation, "export": None, "imports": [], "errors": []}
    if dry_run or not run_validation: return result
    try:
        from nbdev.export import nb_export
        exported = []
        for nb_path in sorted((Path(dest) / "nbs").rglob("*.ipynb")):
            nb_export(str(nb_path), str(dest))
            exported.append(str(nb_path))
        result["export"] = {"returncode": 0, "notebooks": exported, "stdout": "", "stderr": ""}
    except BaseException as exc:
        result["export"] = {"returncode": 1, "stdout": "", "stderr": f"{type(exc).__name__}: {exc}"}
        result["errors"].append("nbdev export failed")
        return result
    env = dict(**os.environ, PYTHONPATH=str(Path(dest).resolve()))
    proc = subprocess.run([sys.executable, "-c", f"import {package}"], cwd=dest, text=True, capture_output=True, env=env)
    result["imports"].append({"module": package, "returncode": proc.returncode, "stderr": proc.stderr[-4000:]})
    if proc.returncode: result["errors"].append(f"import {package} failed")
    return result

In [0]:
#| export
@call_parse
@tracked_call
def py2nbdev(
    source: str,  # Existing Python project/package root
    dest: str,  # Destination nbdev project root
    package: str | None = None,  # Package name to convert
    nbs_path: str = "nbs",  # Notebook folder inside dest
    dry_run: bool = True,  # Plan without writing by default
    force: bool = False,  # Overwrite existing scaffold/notebooks
    run_validation: bool = True,  # Run nbdev export and import checks after writing
):
    "Create a pragmatic nbdev project from a pure-Python package."
    package_name = _primary_package_name(source, package=package)
    dest_root = Path(dest)
    notebook_root = dest_root / nbs_path
    scaffold = [
        _write_if_allowed(dest_root / "pyproject.toml", _nbdev_pyproject_text(package_name, package_name, nbs_path), dry_run=dry_run, force=force),
        _write_if_allowed(dest_root / nbs_path / "index.ipynb", json.dumps(new_nb([mk_cell(f"# {package_name}", cell_type="markdown")]), indent=2), dry_run=dry_run, force=force),
    ]
    conversion = _py2nbs_folder(
        source, nbs_path=str(notebook_root), package=package_name, dry_run=dry_run,
        force=force, skip_init=True, include_tests=False,
    )
    validation = _validate_nbdev_project(dest_root, package_name, run_validation=run_validation, dry_run=dry_run)
    report = {
        "source": str(source),
        "dest": str(dest_root),
        "package": package_name,
        "dry_run": dry_run,
        "scaffold": scaffold,
        "conversion": conversion,
        "validation": validation,
        "blockers": conversion.get("blockers", []) + validation.get("errors", []),
    }
    print(json.dumps(report, indent=2, sort_keys=True, default=str))
    return cli_return(report)